# Tentativo 19 · 4. I modelli lineari

Per i due compiti, con lo stesso split (per voto): **TF-IDF**, **Qwen3 8B** e **Mistral 24B** (strato 27) congelati, con sopra una
**logistica multinomiale** a classi bilanciate (classificazione, 5 livelli) o una **ridge** (regressione, voto 0–100).

**La penalità** si sceglie in ogni fold con 3 fold interni sulle sole testate di training (divise per voto, come lo
split esterno), con la misura per testata: F1 macro sul voto degli articoli (classificazione), Spearman tra voto vero e media
prevista (regressione). Le griglie hanno un ordine di grandezza in più del tentativo 18, dove la scelta cadeva sul bordo;
qui sotto si vede se ci cade ancora.

**Si salvano**, per ogni articolo di ogni fold, i logit (e le probabilità) o il voto previsto, e il modello di ogni fold
(preparazione + modello lineare).

In [ ]:
COMPITI = "classificazione,regressione"
NOMI = "TF-IDF,Qwen3 8B,Mistral 24B"
N_JOBS = 16

In [ ]:
import time
import warnings

import numpy as np
import pandas as pd

import modelli as Mo

warnings.filterwarnings("ignore")
OUT = Mo.Path("Results"); OUT.mkdir(exist_ok=True)
FILE_EMB = {"Qwen3 8B": "Qwen3-Embedding-8B.npy", "Mistral 24B": "Mistral-Small-24B_strato27.npy"}
print(Mo.ambiente())
def registra(msg, file=OUT / "avanzamento.log"):
    riga = f"{time.strftime('%Y-%m-%d %H:%M:%S')}  {msg}"
    print(riga, flush=True)
    with open(file, "a") as f:
        f.write(riga + "\n")

In [ ]:
for compito in COMPITI.split(","):
    art, t = Mo.leggi()
    for nome in NOMI.split(","):
        X = None
        if nome != "TF-IDF":
            ordine = pd.read_csv(Mo.EMBEDDING / "ordine_articoli.csv").article_id
            assert (ordine.to_numpy() == art.article_id.to_numpy()).all()
            X = np.load(Mo.EMBEDDING / FILE_EMB[nome])
        for k in range(1, 6):
            Mo.lineare_fold(compito, nome, art, k, X=X, n_jobs=N_JOBS, registra=registra)

## Le penalità scelte

In [ ]:
righe = []
for compito in COMPITI.split(","):
    for nome in NOMI.split(","):
        for f in sorted((Mo.PREVISIONI / compito / nome).glob("penalita_fold*.csv")):
            p = pd.read_csv(f); g = Mo.GRIGLIE[(compito, "TF-IDF" if nome == "TF-IDF" else "embedding")]
            s = p[p.scelto].iloc[0]
            righe.append({"compito": compito, "modello": nome, "fold": s.fold, "penalità": s.valore,
                          "misura interna": round(s["misura interna"], 3), "sul bordo": s.valore in (g[0], g[-1])})
pen = pd.DataFrame(righe); pen.to_csv(OUT / "penalita_scelte.csv", index=False)
pen.pivot_table(index=["compito", "modello"], columns="fold", values="penalità")

In [ ]:
print("scelte sul bordo della griglia:", int(pen["sul bordo"].sum()), "su", len(pen))
pen[pen["sul bordo"]]